# Radial Basis Functions

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/interpolation_and_curve_fitting/interpolation/radial_basis_functions.ipynb)

Radial basis functions (RBF) are an n-dimensional interpolation technique that doesn't rely on polynomials. Rather, we define a radial basis function, called a *kernel*, applied to each data point:
$$ \varphi_i(||x-x_i||) $$
Commonly, we say $\varphi_i(x=x_i)\equiv 1$. The kernel only depends on the Euclidean distance between the associated data point $x_i$ and the evaluation point $x$.

The interpolation function $y(x)$ is the weighted sum of the $N$ kernels:
$$ y(x) = \sum_{j=1}^N \omega_j \varphi(||x-x_j||) $$

To determine the weights $\omega_j$, we evaluate the function at each data point $x_i$:
$$ y(x_i) = \sum_{j=1}^N \omega_j \varphi(||x_i-x_j||) = y_i $$

Applying this to all $N$ data points generates a linear system to solve for $\omega_j$:
$$
\begin{bmatrix}
\varphi(\|x_1 - x_1\|) & \varphi(\|x_1 - x_2\|) & \cdots & \varphi(\|x_1 - x_N\|) \\
\varphi(\|x_2 - x_1\|) & \varphi(\|x_2 - x_2\|) & \cdots & \varphi(\|x_2 - x_N\|) \\
\vdots & \vdots & \ddots & \vdots \\
\varphi(\|x_N - x_1\|) & \varphi(\|x_N - x_2\|) & \cdots & \varphi(\|x_N - x_N\|)
\end{bmatrix}  \begin{bmatrix}
\omega_1 \\
\omega_2 \\
\vdots \\
\omega_N
\end{bmatrix} = \begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
$$

Kernels are defined with $r = \| x-x_i\|$ and a tuning parameter $\epsilon$. Common kernels include Gaussian ($e^{-\epsilon^2 r^2}$), Inverse quadratic ($\frac{1}{1+[\epsilon r ]^2}$), and Inverse multiquadric ($\frac{1}{\sqrt{1+[\epsilon r ]^2}}$). A good rule of thumb for optimal $\epsilon$ is $\epsilon = \frac{1}{\text{avg} \|x_i-x_j\|}$.

Let's see the kernels:

In [ ]:
import numpy as np
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

# Define the radial basis functions
def gaussian(r, epsilon):
  return np.exp(-(epsilon * r)**2)

def inverse_quadratic(r, epsilon):
  return 1 / (1 + (epsilon * r)**2)

def inverse_multiquadric(r, epsilon):
  return 1 / np.sqrt(1 + (epsilon * r)**2)

epsilon = 1
r_values = np.linspace(0, 10, 100)

fig = go.Figure()
fig.add_trace(go.Scatter(x=r_values, y=gaussian(r_values, epsilon), name='Gaussian'))
fig.add_trace(go.Scatter(x=r_values, y=inverse_quadratic(r_values, epsilon), name='Inverse Quadratic'))
fig.add_trace(go.Scatter(x=r_values, y=inverse_multiquadric(r_values, epsilon), name='Inverse Multiquadric'))
fig.update_layout(title='Radial Basis Functions (epsilon = 1)', xaxis_title='r', yaxis_title='φ(r)')
fig.show()

## 1D RBF Interpolation

For these kernels, the influence of a data point diminishes the further from them you get. 

> In general, $\varphi_i(r=0)$ is not necessarily $1$, and $\varphi(r \rightarrow \infty) \ne 0$, but this requires one more key factor to implement robustly.

**Example - Gaussian sampled at 11 points, equally spaced**

In [ ]:
def f(x):
  return np.exp(-(x/2)**2)

x_toy = np.linspace(-6, 6, 100)
y_toy = f(x_toy)

x_d = np.arange(-5, 6, 1)
y_d = f(x_d)
epsilon = 1

phi_matrix = np.zeros((len(x_d), len(x_d)))
for i in range(len(x_d)):
  for j in range(len(x_d)):
    phi_matrix[i, j] = gaussian(np.abs(x_d[i] - x_d[j]), epsilon)

weights = np.linalg.solve(phi_matrix, y_d)
y_fit = [np.sum([weights[i] * gaussian(np.abs(x - x_d[i]), epsilon) for i in range(len(x_d))]) for x in x_toy]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_toy, y=y_toy, name='Original Function'))
fig.add_trace(go.Scatter(x=x_d, y=y_d, mode='markers', name='Data Points', marker=dict(color='red', size=8)))
fig.add_trace(go.Scatter(x=x_toy, y=y_fit, name='Interpolation', line=dict(dash='dash')))
fig.update_layout(title='Radial Basis Function Interpolation (Gaussian Kernel)', xaxis_title='x', yaxis_title='y')
fig.show()

## 2D RBF Interpolation

Note this is a great result, but it works because the true function tends to zero outside of the data samples. Let's see how RBF interpolation performs on a 2D surface.

In [ ]:
def f(x, y):
    return np.exp(-.5*x**2 - .2*y**2) * np.sin(x) * np.cos(.5*y)

x_grid_vals = np.linspace(-3, 3, 100)
y_grid_vals = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_grid_vals, y_grid_vals)
Z_true = f(X, Y)

num_samples = 100
np.random.seed(42)
x_samples = np.random.uniform(-3, 3, num_samples)
y_samples = np.random.uniform(-3, 3, num_samples)
z_samples = f(x_samples, y_samples)

dist_matrix = np.sqrt((x_samples[:, np.newaxis] - x_samples)**2 + (y_samples[:, np.newaxis] - y_samples)**2)
eps = 1 / np.mean(dist_matrix)
print(f"Using epsilon = {eps:.4f}")

phi_matrix = gaussian(dist_matrix, eps)
print(f'The matrix condition number is {np.linalg.cond(phi_matrix):.2e}')

weights = np.linalg.solve(phi_matrix, z_samples)
Z_fit = np.zeros_like(X)
for i in range(num_samples):
    r = np.sqrt((X - x_samples[i])**2 + (Y - y_samples[i])**2)
    Z_fit += weights[i] * gaussian(r, eps)

surface_fit = go.Surface(x=X, y=Y, z=Z_fit, colorscale='Viridis', showscale=False, name='RBF Fit')
scatter_samples = go.Scatter3d(
    x=x_samples, y=y_samples, z=z_samples,
    mode='markers', marker=dict(color='red', size=3), name='Samples'
)

fig_fit = go.Figure(data=[surface_fit, scatter_samples])
fig_fit.update_layout(
    title='RBF Interpolated Fit and Data Samples',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z', zaxis=dict(range=[-.7, .7]), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.5))
)
fig_fit.show()